# PCA + Ridge (PCR) Volatility Backtest

CPU-only walk-forward backtest using PCA-compressed raw lag features
instead of HAR aggregates. PCA is re-fit every 240 steps; Ridge is
re-fit at the same cadence.

In [ ]:
# ── Setup: clone repo, install deps ──
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!pip install -q numpy pandas scikit-learn pyarrow numba tqdm

In [ ]:
# ── Parameters ──
DATA_PATH = "all30min"
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
N_COMPONENTS = 5

PERIODS_PER_DAY = 48
TRAIN_WINDOW = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
print(f"Train window: {TRAIN_WINDOW} periods ({TRAIN_WINDOW_DAYS} days)")
print(f"PCA components: {N_COMPONENTS}")
print(f"Horizon: {HORIZON}")

In [ ]:
# export
"""PCA + Ridge (PCR) walk-forward backtest for volatility forecasting."""

import argparse
import os

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from tqdm import tqdm

from src.loading import apply_overnight_fills, load_raw_data, parse_exog_cols
from src.scaling import RollingRobustScaler
from src.transforms import (
    PERIODS_PER_DAY,
    SEGMENT_CHOICES,
    SEGMENT_DEFINITIONS,
    compute_segment_train_window,
    generate_raw_lag_features,
    resolve_pca_lags,
    robust_transform,
    slice_to_segment,
)

In [ ]:
# Quick check: imports resolved and segment choices available
print("SEGMENT_CHOICES:", SEGMENT_CHOICES)
print("PERIODS_PER_DAY:", PERIODS_PER_DAY)
print("parse_exog_cols(None):", parse_exog_cols(None))
print("parse_exog_cols('vix|sentiment'):", parse_exog_cols("vix|sentiment"))

In [ ]:
# ── Load + transform ──
import numpy as np
import pandas as pd

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH)
print(f"Loaded: {df.shape}")
print(f"Date range: {df['t'].min()} → {df['t'].max()}")

adj_series, baseline = robust_transform(df, "RV", is_target=True)
df["adj_RV"] = adj_series
df["baseline"] = baseline

print("\nadj_RV stats:")
display(df["adj_RV"].describe())

In [ ]:
# ── Log-spaced raw lag features ──
from src.transforms import generate_raw_lag_features, resolve_pca_lags

lags = resolve_pca_lags()
print(f"Number of lags: {len(lags)}")
print(f"Lags: {lags}")

# Compare with HAR lags
har_lags = [1, 5, 22]  # standard HAR
import matplotlib.pyplot as plt  # noqa: E402

fig, ax = plt.subplots(figsize=(10, 3))
ax.scatter(lags, [1] * len(lags), marker="|", s=200, label="PCR lags (log-spaced)")
ax.scatter(har_lags, [0.8] * len(har_lags), marker="|", s=200, color="red", label="HAR lags")
ax.set_xscale("log")
ax.set_xlabel("Lag (periods)")
ax.set_yticks([])
ax.legend()
ax.set_title("Lag distribution: PCR vs HAR")
plt.tight_layout()
plt.show()

# Generate features
df, feature_names = generate_raw_lag_features(df, target_col="adj_RV")
print(f"\nFeature columns ({len(feature_names)}): {feature_names[:5]} ... {feature_names[-3:]}")
print(f"DataFrame shape: {df.shape}")

In [ ]:
# ── PCA transform: fit on sample, explained variance, scree plot ──
from sklearn.decomposition import PCA

# Horizon shift + drop NaN
df["target"] = df["adj_RV"].shift(-HORIZON)
max_lag = lags[-1]
df_clean = df.iloc[max_lag:].dropna(subset=["target"] + feature_names).reset_index(drop=True)
print(f"Clean rows: {len(df_clean):,}")

# Fit PCA on first train_window rows for inspection
X_sample = df_clean[feature_names].values[:TRAIN_WINDOW].astype(np.float64)
# Simple z-score for visualisation only
X_sample_z = (X_sample - X_sample.mean(axis=0)) / (X_sample.std(axis=0) + 1e-12)

pca_full = PCA(svd_solver="randomized")
pca_full.fit(X_sample_z)

evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)

print(f"\nExplained variance ratio (first {N_COMPONENTS}): {evr[:N_COMPONENTS].round(4)}")
print(f"Cumulative at {N_COMPONENTS} components: {cum_evr[N_COMPONENTS - 1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(1, len(evr) + 1), evr)
ax1.set_xlabel("Component")
ax1.set_ylabel("Explained variance ratio")
ax1.set_title("Scree plot")
ax1.axvline(N_COMPONENTS, color="red", ls="--", label=f"n_components={N_COMPONENTS}")
ax1.legend()

ax2.plot(range(1, len(cum_evr) + 1), cum_evr, "o-")
ax2.axhline(0.95, color="grey", ls="--", alpha=0.5, label="95%")
ax2.axvline(N_COMPONENTS, color="red", ls="--", label=f"n_components={N_COMPONENTS}")
ax2.set_xlabel("Component")
ax2.set_ylabel("Cumulative explained variance")
ax2.set_title("Cumulative variance")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# export
class PCATransform:
    """Thin wrapper around sklearn PCA for the backtest loop."""

    def __init__(self, n_components=5, random_state=42):
        self.pca = PCA(n_components=n_components, svd_solver="randomized", random_state=random_state)

    def fit(self, X, y=None):
        self.pca.fit(X)
        return self

    def transform(self, X):
        return self.pca.transform(X)

In [ ]:
# Smoke-test PCATransform on a tiny random matrix
_rng = np.random.default_rng(0)
_X_toy = _rng.standard_normal((100, 10))
_pca_t = PCATransform(n_components=3).fit(_X_toy)
_out = _pca_t.transform(_X_toy)
print("PCATransform output shape:", _out.shape)
assert _out.shape == (100, 3)

In [ ]:
# export
def run_pcr_backtest(X, y, train_window, n_components=5, refit_frequency=240, alpha=1.0, random_state=42):
    """Walk-forward PCA + Ridge backtest."""
    N, p = X.shape
    n_test = N - train_window
    forecasts = np.empty(n_test, dtype=np.float64)

    scaler = RollingRobustScaler(train_window, p)
    scaler.initialize(X[:train_window])

    pca = PCATransform(n_components=n_components, random_state=random_state)
    X_buf_scaled = scaler.transform_buffer()
    pca.fit(X_buf_scaled)

    X_buf_pca = pca.transform(X_buf_scaled)
    y_buf = y[:train_window]
    ridge = Ridge(alpha=alpha, fit_intercept=True, random_state=random_state)
    ridge.fit(X_buf_pca, y_buf)

    steps_since_refit = 0

    for i in tqdm(range(n_test), desc="PCR backtest", leave=False):
        idx = train_window + i
        x_t = X[idx]

        x_scaled = scaler.transform_single(x_t)
        x_pca = pca.transform(x_scaled.reshape(1, -1))
        forecasts[i] = ridge.predict(x_pca)[0]

        scaler.update(x_t)
        steps_since_refit += 1

        if steps_since_refit >= refit_frequency:
            X_buf_scaled = scaler.transform_buffer()
            pca.fit(X_buf_scaled)
            X_buf_pca = pca.transform(X_buf_scaled)
            buf_start = idx + 1 - train_window
            y_buf = y[buf_start : idx + 1]
            ridge.fit(X_buf_pca, y_buf)
            steps_since_refit = 0

    return forecasts

In [ ]:
# --- PCA + Ridge walk-forward with periodic PCA refit ---
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from tqdm import tqdm
from src.scaling import RollingRobustScaler


class PCATransform:
    """Thin PCA wrapper for the backtest loop."""

    def __init__(self, n_components=5):
        self.pca = PCA(n_components=n_components, svd_solver="randomized")

    def fit(self, X, y=None):
        self.pca.fit(X)
        return self

    def transform(self, X):
        return self.pca.transform(X)


X = df_clean[feature_names].values.astype(np.float64)
y = df_clean["target"].values.astype(np.float64)
dates = df_clean["t"].values
baselines = df_clean["baseline"].values

print(f"X shape: {X.shape}")
print(f"Test samples: {len(y) - TRAIN_WINDOW:,}")

# Walk-forward PCR backtest
N, p = X.shape
n_test = N - TRAIN_WINDOW
forecasts = np.empty(n_test, dtype=np.float64)

scaler = RollingRobustScaler(TRAIN_WINDOW, p)
scaler.initialize(X[:TRAIN_WINDOW])

pca = PCATransform(n_components=N_COMPONENTS)
X_buf_scaled = scaler.transform_buffer()
pca.fit(X_buf_scaled)

X_buf_pca = pca.transform(X_buf_scaled)
y_buf = y[:TRAIN_WINDOW]
ridge = Ridge(alpha=1.0, fit_intercept=True)
ridge.fit(X_buf_pca, y_buf)

steps_since_refit = 0

for i in tqdm(range(n_test), desc="PCR backtest", leave=False):
    idx = TRAIN_WINDOW + i
    x_t = X[idx]

    x_scaled = scaler.transform_single(x_t)
    x_pca = pca.transform(x_scaled.reshape(1, -1))
    forecasts[i] = ridge.predict(x_pca)[0]

    scaler.update(x_t)
    steps_since_refit += 1

    if steps_since_refit >= 240:
        X_buf_scaled = scaler.transform_buffer()
        pca.fit(X_buf_scaled)
        X_buf_pca = pca.transform(X_buf_scaled)
        buf_start = idx + 1 - TRAIN_WINDOW
        y_buf = y[buf_start : idx + 1]
        ridge.fit(X_buf_pca, y_buf)
        steps_since_refit = 0

print(f"\nForecasts: {len(forecasts):,}")

In [ ]:
# ── Quick eval ──
y_test = y[TRAIN_WINDOW:]
dates_test = dates[TRAIN_WINDOW:]
baselines_test = baselines[TRAIN_WINDOW:]

# Adjusted-scale metrics
errors = y_test - forecasts
mse = np.mean(errors**2)
mae = np.mean(np.abs(errors))
print(f"Adjusted-scale MSE: {mse:.6f}")
print(f"Adjusted-scale MAE: {mae:.6f}")

# Duan smearing
smear = np.mean((y_test - forecasts) ** 2)
pred_raw = (forecasts**2 + smear) * baselines_test
true_raw = (y_test**2) * baselines_test

# QLIKE
mask = (true_raw > 0) & (pred_raw > 0)
ratio = true_raw[mask] / pred_raw[mask]
qlike = np.mean(ratio - np.log(ratio) - 1.0)
print(f"QLIKE (raw scale): {qlike:.6f}")

# Save results
results = pd.DataFrame(
    {
        "date": dates_test,
        "horizon": HORIZON,
        "true_adj": y_test,
        "pred_adj": forecasts,
        "true_raw": true_raw,
        "pred_raw": pred_raw,
    }
)
out_path = f"results_pcr_h{HORIZON}.csv"
results.to_csv(out_path, index=False)
print(f"\nSaved {len(results)} rows to {out_path}")
display(results.head())

In [ ]:
# export
def _run_backtest_and_save(
    df: pd.DataFrame,
    feature_names: list[str],
    train_window: int,
    horizon: int,
    start: int,
    end: int,
    output_file: str,
    n_components: int = 5,
    random_state: int = 42,
) -> None:
    """Run PCR backtest on a prepared DataFrame and save results."""
    max_lag = resolve_pca_lags()[-1]

    df["target"] = df["adj_RV"].shift(-horizon)
    df = df.iloc[max_lag:].reset_index(drop=True)
    df = df.dropna(subset=["target"] + feature_names).reset_index(drop=True)

    actual_end = len(df) if end == -1 else end
    df_chunk = df.iloc[start:actual_end].reset_index(drop=True)

    X = df_chunk[feature_names].values.astype(np.float64)
    y = df_chunk["target"].values.astype(np.float64)
    dates = df_chunk["t"].values
    baselines_arr = df_chunk["baseline"].values

    forecasts = run_pcr_backtest(
        X,
        y,
        train_window=train_window,
        n_components=n_components,
        refit_frequency=240,
        random_state=random_state,
    )

    y_test = y[train_window:]
    dates_test = dates[train_window:]
    baselines_test = baselines_arr[train_window:]

    smear = np.mean((y_test - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines_test
    true_raw = (y_test**2) * baselines_test

    results = pd.DataFrame(
        {
            "date": dates_test,
            "horizon": horizon,
            "true_adj": y_test,
            "pred_adj": forecasts,
            "true_raw": true_raw,
            "pred_raw": pred_raw,
        }
    )

    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    results.to_csv(output_file, index=False)
    print(f"Saved {len(results)} rows to {output_file}")

In [ ]:
# Smoke-test _run_backtest_and_save on a tiny slice of the already-loaded data.
# Uses df/feature_names built above; writes to /tmp so notebook state is not polluted.
_small_tw = 200
_small_end = _small_tw + 50  # just 50 test rows
_tmp_out = "/tmp/pcr_smoke.csv"
_run_backtest_and_save(
    df.copy(),
    feature_names,
    train_window=_small_tw,
    horizon=HORIZON,
    start=0,
    end=_small_end,
    output_file=_tmp_out,
    n_components=N_COMPONENTS,
)
_smoke = pd.read_csv(_tmp_out)
print("smoke rows:", len(_smoke))
display(_smoke.head())

In [ ]:
# export
def main() -> None:
    parser = argparse.ArgumentParser(description="PCA + Ridge (PCR) volatility backtest")
    parser.add_argument("--data-path", type=str, required=True)
    parser.add_argument("--horizon", type=int, default=1)
    parser.add_argument("--train-window", type=int, default=500, help="Training window in days")
    parser.add_argument("--start", type=int, default=0)
    parser.add_argument("--end", type=int, default=-1)
    parser.add_argument("--output-file", type=str, default="results_pcr.csv")
    parser.add_argument("--exog-cols", default=None, help="Pipe-separated exog columns, e.g. vix|sentiment")
    parser.add_argument("--segment", default=None, choices=SEGMENT_CHOICES, help="Time-of-day segment")
    parser.add_argument(
        "--lag-scope", default="global", choices=["global", "intra"], help="Compute lags on full dataset or per-segment"
    )
    parser.add_argument("--n-components", type=int, default=5)
    parser.add_argument("--seed", type=int, default=42, help="random_state for randomized PCA / Ridge")
    args = parser.parse_args()

    exog_cols = parse_exog_cols(args.exog_cols)

    # --- Load and transform ---
    df = load_raw_data(args.data_path, allow_missing=True)
    if exog_cols:
        apply_overnight_fills(df, exog_cols)
        df = df.dropna(subset=["RV"] + exog_cols).reset_index(drop=True)

    adj_series, baseline = robust_transform(df, "RV", is_target=True)
    df["adj_RV"] = adj_series
    df["baseline"] = baseline

    adj_exog_cols: list[str] = []
    for col in exog_cols:
        adj_col = f"adj_{col}"
        adj_s, _ = robust_transform(df, col, use_transform=True, use_diurnal=True)
        df[adj_col] = adj_s
        adj_exog_cols.append(adj_col)

    # --- No segment: global backtest ---
    if args.segment is None:
        train_window = args.train_window * PERIODS_PER_DAY
        df, feature_names = generate_raw_lag_features(df, target_col="adj_RV", exog_cols=adj_exog_cols)
        _run_backtest_and_save(
            df,
            feature_names,
            train_window,
            args.horizon,
            args.start,
            args.end,
            args.output_file,
            args.n_components,
            random_state=args.seed,
        )
        return

    # --- Segmented backtest ---
    segments = list(SEGMENT_DEFINITIONS) if args.segment == "all" else [args.segment]

    if args.lag_scope == "global":
        df, feature_names = generate_raw_lag_features(df, target_col="adj_RV", exog_cols=adj_exog_cols)

    for seg_name in segments:
        seg_df = slice_to_segment(df, seg_name)
        if seg_df.empty:
            print(f"No data for segment '{seg_name}'. Skipping.")
            continue

        if args.lag_scope == "intra":
            seg_df, feature_names = generate_raw_lag_features(seg_df, target_col="adj_RV", exog_cols=adj_exog_cols)

        train_window = compute_segment_train_window(seg_df["t"], args.train_window)

        base, ext = os.path.splitext(args.output_file)
        seg_output = f"{base}_{seg_name}{ext}"

        print(f"{'=' * 20} SEGMENT: {seg_name.upper()} {'=' * 20}")
        print(f"Window: {train_window} periods ({args.train_window} days)")
        _run_backtest_and_save(
            seg_df,
            feature_names,
            train_window,
            args.horizon,
            args.start,
            args.end,
            seg_output,
            args.n_components,
            random_state=args.seed,
        )

In [ ]:
# Argparse wiring check: build parser the same way main() does and parse a minimal invocation.
# Avoids running a full backtest from the notebook.
import argparse as _argparse

_p = _argparse.ArgumentParser(description="PCA + Ridge (PCR) volatility backtest")
_p.add_argument("--data-path", type=str, required=True)
_p.add_argument("--horizon", type=int, default=1)
_p.add_argument("--train-window", type=int, default=500)
_p.add_argument("--start", type=int, default=0)
_p.add_argument("--end", type=int, default=-1)
_p.add_argument("--output-file", type=str, default="results_pcr.csv")
_p.add_argument("--exog-cols", default=None)
_p.add_argument("--segment", default=None, choices=SEGMENT_CHOICES)
_p.add_argument("--lag-scope", default="global", choices=["global", "intra"])
_p.add_argument("--n-components", type=int, default=5)
_args = _p.parse_args(["--data-path", "all30min", "--horizon", "1", "--segment", "open"])
print(_args)
assert _args.segment == "open"
assert _args.lag_scope == "global"

In [ ]:
# export
if __name__ == "__main__":
    main()

In [ ]:
from _exporter import export_notebook

export_notebook("ml_pcr.ipynb", "../src/ml_pcr.py")